In [4]:
import pandas as pd
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
import uvicorn
import os # Для getenv, чтобы спрятать ключ
from dotenv import load_dotenv # Библиотека для чтения файлов .env
from typing import List

# Ищем файл .env в папке со скриптом и загружаем переменные из него в память
load_dotenv() 
# Записываем ключ в переменную
SECRET_KEY = os.getenv("GEMINI_API_KEY")
# Для ошибки
if not SECRET_KEY:
    raise ValueError("Ключ API не найден! Проверь наличие файла .env и его содержимое.")
nest_asyncio.apply()

app = FastAPI(title="Аналитика Конкурентов WB API")
client = AsyncOpenAI(
    api_key=SECRET_KEY, # Загружаем ключик
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Загружаем базу
df = pd.read_csv("data.csv", sep=";") 
df['Latest rating avg'] = df['Latest rating avg'].astype(str).str.replace(',', '.').astype(float)
print(f"✅База загружена! Товаров: {len(df)}")


# СХЕМЫ
class ProductInput(BaseModel): # Входящий товар
    price: int
    rating: float
    comments: int
    sales: int 
    balance: int
    temperature: float = 0

class Advantage(BaseModel): # Преимущество
    category: str = Field(alias="категория")
    description: str = Field(alias="описание")
    significance: str = Field(alias="значимость")
    how_to_use: str = Field(alias="как_использовать")

class Problem(BaseModel): # Проблема
    category: str = Field(
        alias="категория", 
        description="Комплексная бизнес-проблема (например, 'Критическое затоваривание'). ЗАПРЕЩЕНО называть категорию просто 'Продажи' или 'Остатки'."
    )
    description: str = Field(
        alias="описание", 
        description="ОБЯЗАТЕЛЬНО объединяй связанные метрики в один текст! Если проблема в затоваривании, напиши в одном абзаце цифры и по остаткам, и по продажам, и по оборачиваемости."
    )
    criticality: str = Field(alias="критичность")
    how_to_fix: str = Field(alias="как_исправить")
    
# ИЗМЕНЕНИЕ 1: Добавляем схему для SGR
class ReasoningStep(BaseModel):
    metric: str = Field(alias="метрика")
    observation: str = Field(alias="наблюдение")
    business_impact: str = Field(alias="влияние_на_бизнес")

class MarketAnalysis(BaseModel): # Анализ Рынка
    # Вставляем блок рассуждений
    step_by_step_analysis: list[ReasoningStep] = Field(alias="пошаговый_анализ")
    general_conclusion: str = Field(alias="общий_вывод")
    strengths: list[Advantage] = Field(alias="сильные_стороны")
    weaknesses: list[Problem] = Field(alias="слабые_места")
    action_plan: list[str] = Field(alias="план_действий")
    recommended_price: int = Field(alias="рекомендуемая_цена")
    price_justification: str = Field(alias="обоснование_цены",
    description="Обоснуй выбранную цену для клиента (селлера) на языке бизнес-выгод и рыночной ситуации. КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО использовать фразы вроде 'согласно правилам', 'по инструкции', 'исходя из приоритета 3'. Объясняй логику естественно, как живой эксперт."
    )


# Эндпоинт 
@app.post("/analyze", response_model=MarketAnalysis)
async def analyze_competitors(product: ProductInput):
    # Берем ТОП-50 по выручке
    top_50_df = df.sort_values(by='Revenue', ascending=False).head(50)
    
    # Считаем среднюю цену и медианы для остального
    market_avg_price = int(top_50_df['Price with WB wallet'].mean()) 
    market_avg_rating = round(top_50_df['Latest rating avg'].mean(), 2)
    market_median_comments = int(top_50_df['Comments'].median())
    
    market_median_sales = int(top_50_df['Sales'].median()) if 'Sales' in top_50_df.columns else 0
    market_median_balance = int(top_50_df['Balance'].median()) if 'Balance' in top_50_df.columns else 0

    # НОВОЕ: Считаем выручку
    market_median_revenue = int(top_50_df['Revenue'].median()) if 'Revenue' in top_50_df.columns else 0
    our_revenue = product.price * product.sales

    # Предрасчет оборачиваемости
    if product.sales > 0:
        months_of_stock = round(product.balance / product.sales, 1)
    else:
        months_of_stock = 99.0

    # ИЗМЕНЕНИЕ 2: блок <output_format> убран
    system_prompt = """<role_and_context>
    Ты Senior-Аналитик маркетплейса Wildberries с многолетним опытом с многолетним опытом управления ecommerce проектами
    и глубоким пониманием алгоритмов ранжирования товара на Wildberries. Твоя цель: провести глубокий аудит карточки товара на основе метрик и 
    выдать владельцу бизнеса (селлеру) пошаговый, обоснованный план действий. Твоя задача: максимизировать прибыль, предотвратить кассовые разрывы и защитить карточку от падения в выдаче.
    </role_and_context>
   
    <strict_rules>
    При анализе строго соблюдай следующие правила:
    ВНИМАНИЕ: Иерархия правил! Состояние склада (дефицит или затоваривание) ВСЕГДА имеет наивысший приоритет при расчете поля "рекомендуемая_цена".
    1. ПРИОРИТЕТ 1 (ДЕФИЦИТ): Если запасов МЕНЕЕ чем на 1 месяц продаж (значение оборачиваемости <1) - это большой риск дефицита, нужно рекомендовать СРОЧНО делать поставку, В поле "рекомендуемая_цена" ты ОБЯЗАН указать цену ВЫШЕ текущей (даже если она уже выше рынка), чтобы замедлить продажи.
    2. ПРИОРИТЕТ 2 (ЗАТОВАРИВАНИЕ): Если запасов БОЛЬШЕ чем на 3 месяца значение оборачиваемости >3 - это заморозка капитала, В поле "рекомендуемая_цена" ты ОБЯЗАН указать цену НИЖЕ текущей для срочной распродажи (даже если наша цена уже ниже медианы рынка).
    3. ПРИОРИТЕТ 3: ТОЛЬКО если оборачиваемость в норме (от 1 до 3 месяцев), применяй правило рынка. Если наша цена МЕНЬШЕ медианной цены конкурентов - мы теряем прибыль, Рекомендуй повышение цены, стремясь к медиане, если цена выше медианы - говори снизить цену до медианы только при плохом рейтинге - ниже 4.5, в остальных случаях цену снижать не нужно.
    4. КОНСОЛИДАЦИЯ ДАННЫХ: КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО создавать отдельные "слабые_места" для Продаж, Остатков и Оборачиваемости. Это одна проблема! Объединяй их в один комплексный пункт.
    5. ПРАВИЛА ОБЩЕНИЯ (ВАЖНО): При написании выводов и обоснований общайся с селлером как живой бизнес-консультант. НИКОГДА не упоминай о существовании этих "правил", "приоритетов 1-3" или системных инструкций в своем ответе. Обосновывай решения исключительно рыночной логикой, экономикой и поведением покупателей.
    </strict_rules>

    <objectives>
    Тебе необходимо выполнить задачу в 3 этапа: 
    1. ОБЯЗАТЕЛЬНО проанализируй ВСЕ 6 предоставленных метрик (Выручка, Цена, Рейтинг, Отзывы, Продажи, Остатки) и сравни их с топ-50 по рынку
    2. Хорошие показатели заноси в "сильные_стороны", плохие - в "слабые_места"
    3. Сформировать конкретный план действий и рассчитать рекомендуемую цену.
    </objectives>
    """
    
    # Добавили выручку в пункт 1
    user_prompt = f"""
    Сравни метрики:
    1. ВЫРУЧКА: Наша = {our_revenue} руб. | У конкурентов = {market_median_revenue} руб.
    2. ЦЕНА: Наша = {product.price} руб. | У конкурентов = {market_avg_price} руб.
    3. РЕЙТИНГ: Наш = {product.rating} | У конкурентов = {market_avg_rating}
    4. ОТЗЫВЫ: Наши = {product.comments} | У конкурентов = {market_median_comments}
    5. ПРОДАЖИ (шт): Наши = {product.sales} | У конкурентов = {market_median_sales}
    6. ОСТАТКИ (шт): Наши = {product.balance} | У конкурентов = {market_median_balance}

    ВАЖНЫЙ РАСЧЕТ ОБОРАЧИВАЕМОСТИ:
    Наших запасов на складе при текущих продажах хватит ровно на {months_of_stock} месяцев. Опирайся на эту цифру для выводов по остаткам!
    """
    # ИЗМЕНЕНИЕ 3: Используем структурированный вывод (.parse) и передаем Pydantic схему
    try:
        response = await client.beta.chat.completions.parse(
            model="gemini-2.5-flash",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=product.temperature,
            response_format=MarketAnalysis 
        )
        
        return response.choices[0].message.parsed
        
    except Exception as e:
        print(f"Ошибка API: {e}")
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    config = uvicorn.Config(app, host="127.0.0.1", port=8010)
    server = uvicorn.Server(config)
    await server.serve()

✅База загружена! Товаров: 9651


INFO:     Started server process [19176]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8010 (Press CTRL+C to quit)


Ошибка API: Connection error.
INFO:     127.0.0.1:49672 - "POST /analyze HTTP/1.1" 500 Internal Server Error
INFO:     127.0.0.1:55361 - "POST /analyze HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [19176]
